In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np

path = '/kaggle/input/datasets/nipunarora8/age-gender-and-ethnicity-face-data-csv/age_gender.csv'

df = pd.read_csv(path)

print(df.shape)
print(df.columns)
df.head()

In [ ]:
print(df.isnull().sum())

print("Gender:", np.unique(df['gender']))
print("Ethnicity:", np.unique(df['ethnicity']))
print("Age:", df['age'].min(), "to", df['age'].max())

In [ ]:
X = np.array(
    df['pixels'].apply(
        lambda x: np.array(x.split(), dtype=np.float32)
    ).tolist()
)

print(X.shape)

In [ ]:
X = X.reshape(-1, 48, 48, 1)

print(X.shape)

In [ ]:
X = np.repeat(X, 3, axis=-1)

print(X.shape)

In [ ]:
y_age = df['age'].values
y_gender = df['gender'].values
y_ethnicity = df['ethnicity'].values

In [ ]:
print("Gender classes:", np.unique(y_gender))
print("Ethnicity classes:", np.unique(y_ethnicity))

In [ ]:
num_ethnicity_classes = len(np.unique(y_ethnicity))

print("Number of ethnicity classes:", num_ethnicity_classes)

In [ ]:
from sklearn.model_selection import train_test_split

# 80% train+validation, 20% test
(
    X_train,
    X_test,
    y_age_train,
    y_age_test,
    y_gender_train,
    y_gender_test,
    y_ethnicity_train,
    y_ethnicity_test
) = train_test_split(
    X,
    y_age,
    y_gender,
    y_ethnicity,
    test_size=0.2,
    random_state=42,
    stratify=y_gender
)

# Split the 80% into ~70% train and 10% validation
(
    X_train,
    X_val,
    y_age_train,
    y_age_val,
    y_gender_train,
    y_gender_val,
    y_ethnicity_train,
    y_ethnicity_val
) = train_test_split(
    X_train,
    y_age_train,
    y_gender_train,
    y_ethnicity_train,
    test_size=0.125,
    random_state=42,
    stratify=y_gender_train
)

In [ ]:
X_train.shape

In [ ]:
X_val.shape

In [ ]:
X_test.shape

In [ ]:
from sklearn.preprocessing import StandardScaler

age_scaler = StandardScaler()

age_train_scaled = age_scaler.fit_transform(y_age_train.reshape(-1,1))
age_val_scaled = age_scaler.transform(y_age_val.reshape(-1,1))
age_test_scaled = age_scaler.transform(y_age_test.reshape(-1,1))

In [ ]:
from keras.applications import EfficientNetB0

base_model = EfficientNetB0(
    weights = 'imagenet',
    include_top = False,
    input_shape = (224,224,3)
)

In [ ]:
base_model.trainable = False

In [ ]:
from keras.layers import Input,Dense,Dropout,Resizing,GlobalAveragePooling2D
from keras import Model

In [ ]:
inputs = Input(shape=(48,48,3))
X = Resizing(224,224)(inputs)
X = base_model(X)
X = GlobalAveragePooling2D()(X)
X = Dense(256,activation='relu')(X)
X = Dropout(0.3)(X)

In [ ]:
age_output = Dense(1,
                   activation = 'linear',
                   name='age')(X)
gender_output = Dense(
    1,
    activation='sigmoid',
    name='gender'
)(X)
num_ethnicity_classes = len(np.unique(y_ethnicity))

ethnicity_output = Dense(
    num_ethnicity_classes,
    activation='softmax',
    name='ethnicity'
)(X)

In [ ]:
model = Model(
    inputs = inputs,
    outputs = [
        age_output,
        gender_output,
        ethnicity_output
    ]
)

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer = 'adam',
    loss = {
        'age' : 'mse',
        'gender' : 'binary_crossentropy',
        'ethnicity' : 'sparse_categorical_crossentropy'
    },
    metrics = {
        'age': ['mae'],
        'gender': ['accuracy'],
        'ethnicity': ['accuracy']
    }
)

In [ ]:
from keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    restore_best_weights = True,
    patience = 3,
    monitor = 'val_loss'
)

In [ ]:
history = model.fit(
    X_train,
    {
        'age' : age_train_scaled,
        'gender' : y_gender_train,
        'ethnicity' : y_ethnicity_train
    },
    validation_data = (
        X_val,
        {
            'age' : age_val_scaled,
            'gender' : y_gender_val,
            'ethnicity' : y_ethnicity_val
        }
    ),
    epochs = 30,
    batch_size = 32,
    callbacks = [early_stop]
)

In [ ]:
age_pred_scaled, gender_pred, ethnicity_pred = model.predict(X_test)

In [ ]:
age_pred = age_scaler.inverse_transform(
    age_pred_scaled
).flatten()

In [ ]:
gender_pred_class = (gender_pred > 0.5).astype(int).flatten()

In [ ]:
import numpy as np

ethnicity_pred_class = np.argmax(
    ethnicity_pred,
    axis=1
)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

age_mae = mean_absolute_error(
    y_age_test,
    age_pred
)

age_rmse = np.sqrt(
    mean_squared_error(
        y_age_test,
        age_pred
    )
)

age_r2 = r2_score(
    y_age_test,
    age_pred
)

print("Age MAE:", age_mae)
print("Age RMSE:", age_rmse)
print("Age R²:", age_r2)

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_gender_test,
        gender_pred_class
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix

print(
    confusion_matrix(
        y_gender_test,
        gender_pred_class
    )
)

In [ ]:
print(
    classification_report(
        y_ethnicity_test,
        ethnicity_pred_class
    )
)

In [ ]:
print(
    confusion_matrix(
        y_ethnicity_test,
        ethnicity_pred_class
    )
)

In [ ]:
print(confusion_matrix(
    y_ethnicity_test,
    ethnicity_pred_class
))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Ethnicity Confusion Matrix')
plt.show()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_ethnicity_train)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_ethnicity_train
)

ethnicity_class_weights = dict(
    zip(classes, weights)
)

print(ethnicity_class_weights)

In [ ]:
model.compile(
    optimizer='adam',

    loss={
        'age': 'mae',
        'gender': 'binary_crossentropy',
        'ethnicity': 'sparse_categorical_crossentropy'
    },

    loss_weights={
        'age': 1.0,
        'gender': 1.0,
        'ethnicity': 1.5
    },

    metrics={
        'age': ['mae'],
        'gender': ['accuracy'],
        'ethnicity': ['accuracy']
    }
)

In [ ]:
from keras import Sequential
from keras.layers import (
    RandomFlip,
    RandomRotation,
    RandomZoom
)

augmentation = Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.05),
    RandomZoom(0.1)
])

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

In [ ]:
import keras
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),

    loss={
        'age': 'mae',
        'gender': 'binary_crossentropy',
        'ethnicity': 'sparse_categorical_crossentropy'
    },

    metrics={
        'age': ['mae'],
        'gender': ['accuracy'],
        'ethnicity': ['accuracy']
    }
)

In [ ]:
history_ft = model.fit(
    X_train,
    {
        'age': age_train_scaled,
        'gender': y_gender_train,
        'ethnicity': y_ethnicity_train
    },
    validation_data=(
        X_val,
        {
            'age': age_val_scaled,
            'gender': y_gender_val,
            'ethnicity': y_ethnicity_val
        }
    ),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop]
)

In [ ]:
age_pred_scaled, gender_pred, ethnicity_pred = model.predict(X_test)

In [ ]:
age_pred = age_scaler.inverse_transform(
    age_pred_scaled
).flatten()

gender_pred_class = (
    gender_pred > 0.5
).astype(int).flatten()

ethnicity_pred_class = np.argmax(
    ethnicity_pred,
    axis=1
)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)
import numpy as np

In [ ]:
age_mae = mean_absolute_error(
    y_age_test,
    age_pred
)

age_rmse = np.sqrt(
    mean_squared_error(
        y_age_test,
        age_pred
    )
)

age_r2 = r2_score(
    y_age_test,
    age_pred
)

print("Age MAE:", age_mae)
print("Age RMSE:", age_rmse)
print("Age R²:", age_r2)

In [ ]:
print("Gender Accuracy:",
      accuracy_score(
          y_gender_test,
          gender_pred_class
      ))

print(
    classification_report(
        y_gender_test,
        gender_pred_class
    )
)

In [ ]:
print("Ethnicity Accuracy:",
      accuracy_score(
          y_ethnicity_test,
          ethnicity_pred_class
      ))

print(
    classification_report(
        y_ethnicity_test,
        ethnicity_pred_class
    )
)

In [ ]:
cm = confusion_matrix(
    y_ethnicity_test,
    ethnicity_pred_class
)

print(cm)

In [ ]:
print(classification_report(
    y_ethnicity_test,
    ethnicity_pred_class
))

In [ ]:
from keras import Sequential
from keras.layers import RandomFlip, RandomRotation, RandomZoom

augmentation = Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.05),
    RandomZoom(0.1)
])

In [ ]:
inputs = Input(shape=(48, 48, 3))

x = augmentation(inputs)

x = Resizing(224, 224)(x)

x = base_model(x)

x = GlobalAveragePooling2D()(x)

x = Dense(256, activation='relu')(x)

x = Dropout(0.3)(x)

In [ ]:
age_output = Dense(
    1,
    activation='linear',
    name='age'
)(x)

gender_output = Dense(
    1,
    activation='sigmoid',
    name='gender'
)(x)

ethnicity_output = Dense(
    num_ethnicity_classes,
    activation='softmax',
    name='ethnicity'
)(x)

In [ ]:
model = Model(
    inputs=inputs,
    outputs=[
        age_output,
        gender_output,
        ethnicity_output
    ]
)

In [ ]:
base_model.trainable = False

In [ ]:
model.compile(
    optimizer='adam',
    loss={
        'age': 'mae',
        'gender': 'binary_crossentropy',
        'ethnicity': 'sparse_categorical_crossentropy'
    },
    metrics={
        'age': ['mae'],
        'gender': ['accuracy'],
        'ethnicity': ['accuracy']
    }
)

In [ ]:
history_aug = model.fit(
    X_train,
    {
        'age': age_train_scaled,
        'gender': y_gender_train,
        'ethnicity': y_ethnicity_train
    },
    validation_data=(
        X_val,
        {
            'age': age_val_scaled,
            'gender': y_gender_val,
            'ethnicity': y_ethnicity_val
        }
    ),
    epochs=30,
    batch_size=32,
    callbacks=[early_stop]
)

In [ ]:
age_pred_scaled, gender_pred, ethnicity_pred = model.predict(X_test)

age_pred = age_scaler.inverse_transform(
    age_pred_scaled
).flatten()

gender_pred_class = (
    gender_pred > 0.5
).astype(int).flatten()

ethnicity_pred_class = np.argmax(
    ethnicity_pred,
    axis=1
)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)
import numpy as np

age_mae = mean_absolute_error(y_age_test, age_pred)
age_rmse = np.sqrt(
    mean_squared_error(y_age_test, age_pred)
)
age_r2 = r2_score(y_age_test, age_pred)

gender_acc = accuracy_score(
    y_gender_test,
    gender_pred_class
)

ethnicity_acc = accuracy_score(
    y_ethnicity_test,
    ethnicity_pred_class
)

print("Age MAE:", age_mae)
print("Age RMSE:", age_rmse)
print("Age R²:", age_r2)

print("\nGender Accuracy:", gender_acc)
print("\nEthnicity Accuracy:", ethnicity_acc)

print("\nEthnicity Report:")
print(classification_report(
    y_ethnicity_test,
    ethnicity_pred_class
))